# Marker Usage

Analyze C1 marker counts and spatial marker heatmaps.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
episodes_path = Path('../runs/processed/episodes.csv')
markers_path = Path('../runs/processed/markers.csv')
episodes = pd.read_csv(episodes_path) if episodes_path.exists() else pd.DataFrame()
markers = pd.read_csv(markers_path) if markers_path.exists() else pd.DataFrame()
episodes.head()

In [ ]:
c1 = episodes[episodes['condition'] == 'C1'] if not episodes.empty else pd.DataFrame()
if not c1.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=c1, x='markers_used', bins=20)
    plt.title('Markers per episode')
    plt.xlabel('Marker placements')
    plt.show()

In [ ]:
if not c1.empty:
    ordered = c1.sort_values(['run_id', 'episode_index']).copy()
    ordered['marker_rate'] = ordered.groupby('run_id')['markers_used'].transform(lambda s: s.rolling(50, min_periods=1).mean())
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=ordered, x='episode_index', y='marker_rate', errorbar='se')
    plt.title('Marker usage over training')
    plt.xlabel('Episode')
    plt.ylabel('Rolling markers per episode')
    plt.show()

In [ ]:
if not markers.empty:
    maps = [np.asarray(json.loads(x), dtype=float) for x in markers['marker_map'].dropna()]
    if maps:
        heatmap = np.sum(maps, axis=0)
        plt.figure(figsize=(6, 5))
        sns.heatmap(heatmap, cmap='mako')
        plt.title('Spatial marker heatmap')
        plt.show()